In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'backend').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'backend').is_dir():
    raise FileNotFoundError('Could not find the HireAI project root. Run this notebook from within the project directory.')
DATASET_PROCESSED = PROJECT_ROOT / "ai" / "datasets" / "processed"

RESULT_FILE = DATASET_PROCESSED / "qwen3_v2_baseline_results.json"
TEST_FILE = DATASET_PROCESSED / "hireai_test_v2.jsonl"

print("Project root:", PROJECT_ROOT)
print("Results file:", RESULT_FILE)
print("Test file:", TEST_FILE)

In [ ]:
with open(
    RESULT_FILE,
    "r",
    encoding="utf-8"
) as f:
    baseline = json.load(f)

print("Model:", baseline["model"])
print("Dataset:", baseline["dataset"])
print("Examples:", baseline["num_examples"])

In [ ]:
test_data = []

with open(
    TEST_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:
        test_data.append(json.loads(line))

print("Test examples:", len(test_data))

In [ ]:
print(baseline["field_scores"])

In [ ]:
print(
    json.dumps(
        baseline["results"][0],
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
job_title_errors = []

for result in baseline["results"]:

    score = result["scores"].get("job_titles")

    if score is not None and score < 1.0:

        job_title_errors.append({
            "index": result["index"],
            "score": score,
            "predicted": result["predicted"]
        })

print(
    "Job title errors:",
    len(job_title_errors)
)

In [ ]:
for error in job_title_errors[:10]:

    print("=" * 70)
    print("INDEX:", error["index"])
    print("SCORE:", error["score"])

    print("\nPREDICTED JOB TITLES:")

    print(
        error["predicted"]["job_titles"]
    )

    print()

In [ ]:
experience_errors = []

for result in baseline["results"]:

    score = result["scores"].get(
        "years_of_experience"
    )

    if score == 0:

        experience_errors.append({
            "index": result["index"],
            "predicted": result["predicted"]
        })

print(
    "Years-of-experience errors:",
    len(experience_errors)
)

In [ ]:
for error in experience_errors[:10]:

    print("=" * 70)
    print("INDEX:", error["index"])

    print(
        "PREDICTED:",
        error["predicted"]["years_of_experience"]
    )

In [ ]:
index = experience_errors[0]["index"]

example = test_data[index]
result = baseline["results"][index]

print("=" * 80)
print("RESUME")
print("=" * 80)

print(example["input"])

print("\n")
print("=" * 80)
print("EXPECTED")
print("=" * 80)

print(
    json.dumps(
        example["output"],
        indent=2,
        ensure_ascii=False
    )
)

print("\n")
print("=" * 80)
print("PREDICTED")
print("=" * 80)

print(
    json.dumps(
        result["predicted"],
        indent=2,
        ensure_ascii=False
    )
)